# 🛡️ 기능 2. 보안 샌드박스 (Defense Arena) - 피어 리뷰, 가설 검증 및 모의 디펜스 (심화)

본 가이드는 **보안 격리 샌드박스**의 핵심 심화 기능인 **3대 에이전트 다자간 피어 리뷰 시뮬레이션, Self-Consistency 기반 가설 검증 (다수결 합의), 턴제 모의 디펜스 실시간 평가 피드백 및 세션 완전 소거(Wipe Out)**의 핵심 동작 원리를 심도 있게 학습합니다.

학습용으로 불필요한 네트워크 대기 및 데이터베이스 세션 CRUD 코드를 제거하고, **실제 에이전트와 체인을 조립하는 핵심 로직**을 중심으로 구성되었습니다.

---

## 📌 심화 학습 핵심 포인트
1. **다자간 에이전트 피어 리뷰 (Multi-Agent Peer Review)**
   - `gpt-4o-mini`와 `with_structured_output` API를 활용하여 Methodology, Novelty, Academic Style 3대 리뷰 에이전트 비평을 DTO 규격으로 추출하는 원리를 이해합니다.
2. **Self-Consistency 가설 검증**
   - RAG 팩트 검색 결과에 대해 온도를 다르게 설정한 3회의 독립 추론 투표를 가동하여 합의 결론(`verdict`) 및 합의율(`Consensus Ratio`)을 도출합니다.
3. **턴제 모의 디펜스 아레나 & 실시간 채점**
   - 3턴 동안 진행되는 면접 로직을 설계하며, 100점 만점 루브릭 채점 모델(`ScoreDTO`)을 통해 질문과 답변을 정밀 피드백하는 흐름을 배웁니다.
4. **수명 주기 관리 및 파쇄 (Wipe Out)**
   - 기밀 정보 영구 소거를 위한 백그라운드 셧다운 및 `shutil.rmtree` 물리 삭제 메커니즘을 확인합니다.

## 1단계. 환경 및 백엔드 설정 로드

필수 환경 설정과 OpenAI 연결을 체크하고 백엔드 패키지 주입을 수행합니다.

In [1]:
import os
import sys
import asyncio
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 현재 notebooks/defense_arena/ 폴더 경로 기준 백엔드 디렉토리 탐색
current_dir = Path("").resolve() if '__file__' not in locals() else Path(__file__).parent
backend_dir = (current_dir / "../../backend").resolve()

if str(backend_dir) not in sys.path:
    sys.path.append(str(backend_dir))

load_dotenv(dotenv_path=backend_dir / ".env")
openai_key = os.getenv("OPENAI_API_KEY", "")
print(f"🔑 OpenAI API Key 설정 완료: {'Yes' if openai_key else 'No'}")

🔑 OpenAI API Key 설정 완료: Yes


## 2단계. 3대 에이전트 다자간 피어 리뷰 시뮬레이션

백엔드 [services.py](file:///Users/pileuszu/Repos/bist-mini-2/backend/api/v1/defense_arena/services.py#L263-L314)의 핵심 로직과 동일하게 Pydantic 구조화 출력을 생성하고, `isinstance` 타입 가드를 적용하여 안전하게 데이터를 파싱합니다.
3대 심사 에이전트:
1. **Methodology**: 방법론적 엄밀함, 실험 설계 비평.
2. **Novelty**: 선행 연구 대비 참신성, SOTA 기여도 분석.
3. **Academic Style**: 학술적 영문 스타일 및 구조적 가독성 감사.

In [2]:
# --- Pydantic 구조 정의 (DTO) ---
class AgentOpinion(BaseModel):
    agent_type: str = Field(description="Agent type: methodology, novelty, or academic_style")
    score: int = Field(description="Score assigned by this agent (0 to 100)")
    feedback: str = Field(description="Detailed critical feedback and recommendations in English")

class PeerReviewReport(BaseModel):
    overall_score: int = Field(description="Weighted average score assigned by the entire committee")
    opinions: list[AgentOpinion] = Field(description="List of opinions from the 3 specialized agents")
    review_report: str = Field(description="Consolidated final review summary report text")

# --- 체인 조립 및 호출 ---
async def simulate_peer_review(document_content: str, target_journal: str) -> PeerReviewReport:
    if not openai_key:
        # API 키가 없을 때의 fallback
        return PeerReviewReport(
            overall_score=75,
            opinions=[
                AgentOpinion(agent_type="methodology", score=70, feedback="Insufficient baseline comparison."),
                AgentOpinion(agent_type="novelty", score=80, feedback="Interesting idea but incremental."),
                AgentOpinion(agent_type="academic_style", score=75, feedback="Typographical errors found in Section 2.")
            ],
            review_report="The overall paper draft is promising but lacks thorough evaluation."
        )

    llm = ChatOpenAI(model="gpt-4o-mini", openai_api_key=openai_key, temperature=0.2)
    structured_llm = llm.with_structured_output(PeerReviewReport)

    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are an elite, highly critical academic peer review committee grading a draft submission to the target journal: '{target_journal}'.\n"
            "Your committee consists of three specialized reviewer agents:\n"
            "1. Methodology Agent: Evaluates the scientific rigor, mathematical proofs, experimental design, baseline comparisons, and validity of assumptions. Be extremely thorough; point out any lack of details or statistical significance.\n"
            "2. Novelty Agent: Critiques the state-of-the-art (SOTA) literature comparison, the uniqueness of the proposed solution, and its theoretical or practical contributions. Determine if the paper is a mere incremental change or a substantial breakthrough.\n"
            "3. Academic Style Agent: Audits structural clarity, logical flow, formatting consistency, clarity of figures/tables descriptions, and use of precise academic English. Identify awkward sentences or passive structures.\n\n"
            "Conduct a rigorous, critical review simulation. Provide highly detailed, professional, and actionable feedback in English.\n"
            "Assign realistic, rigorous scores (out of 100) for each agent's category, and synthesize the overall report summary.\n"
            "Ensure your criticism is academic, specific to the text, and helps the authors successfully revise for the target journal."
        )),
        ("user", "Paper Draft Content:\n{context}")
    ])

    chain = prompt | structured_llm
    report = await chain.ainvoke({
        "target_journal": target_journal,
        "context": document_content
    })

    # 통합 개발 가이드라인의 [LangChain Structured Output 타입 검증] 규칙 준수
    if not isinstance(report, PeerReviewReport):
        raise TypeError(f"Expected PeerReviewReport, got {type(report)}")
        
    return report

# 테스트 실행
sample_draft = """We propose a new lightweight attention mechanism called MicroAttention for mobile vision tasks.
The model reduces Parameters by 30% and keeps accuracy. However, we only compared our method with MobileNetV2 and MobileNetV3.
The experiments were conducted on a subset of ImageNet (10 classes only due to compute limits)."""

report_result = await simulate_peer_review(sample_draft, "IEEE Access")
print(f"📊 종합 점수: {report_result.overall_score}/100")
for op in report_result.opinions:
    print(f"   - [{op.agent_type.upper()}] 점수: {op.score} | 비평: {op.feedback[:100]}...")
print(f"📝 종합 심사 평 요약:\n{report_result.review_report}")

📊 종합 점수: 65/100
   - [METHODOLOGY] 점수: 60 | 비평: The experimental design lacks rigor due to the limited comparison with only MobileNetV2 and MobileNe...
   - [NOVELTY] 점수: 70 | 비평: While the introduction of MicroAttention is a step towards optimizing attention mechanisms for mobil...
   - [ACADEMIC_STYLE] 점수: 75 | 비평: The paper's structure is generally clear, but there are areas where clarity could be improved. For i...
📝 종합 심사 평 요약:
The draft presents a new lightweight attention mechanism, MicroAttention, aimed at mobile vision tasks. However, the submission has several critical weaknesses that need to be addressed before it can be considered for publication in IEEE Access. The methodology lacks robustness due to limited baseline comparisons and insufficient statistical analysis. The novelty of the proposed mechanism is not clearly articulated, and the paper does not adequately position itself within the existing literature. Finally, while the academic style is generally acceptable, im

/Users/pileuszu/Repos/bist-mini-2/backend/venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=PeerReviewReport(overall_...chances of acceptance."), input_type=PeerReviewReport])
  return self.__pydantic_serializer__.to_python(


## 3단계. Self-Consistency 기반 가설 검증

사용자가 제안한 가설을 RAG로 로드된 근거 팩트와 매핑하여 3회 독립 시행 다수결(SUPPORT / REFUTE / INSUFFICIENT_EVIDENCE)로 판단하고 Consensus Ratio(일치율)를 연산합니다.
온도(temperature)를 각각 `0.5`, `0.6`, `0.7`로 다르게 설정하여 다양성을 확보합니다.

In [3]:
class HypothesisVoteItem(BaseModel):
    vote: str = Field(description="Verdict of evaluation: SUPPORT, REFUTE, or INSUFFICIENT_EVIDENCE")
    rationale: str = Field(description="Logical reasoning based strictly on the facts provided")

async def verify_hypothesis_consistency(hypothesis: str, extracted_facts: list[str]) -> dict:
    citations_context = "\n\n".join([f"[Fact {i+1}]: {c}" for i, c in enumerate(extracted_facts)])
    
    detailed_votes = []
    votes_map = {"SUPPORT": 0, "REFUTE": 0, "INSUFFICIENT_EVIDENCE": 0}
    
    if not openai_key:
        return {"verdict": "SUPPORT", "consensus_ratio": 1.0, "details": "Mocked support"}

    for i in range(3):
        # 온도를 다르게 주어 독립 시행을 유도
        temp = 0.5 + (i * 0.1)
        llm = ChatOpenAI(model="gpt-4o-mini", openai_api_key=openai_key, temperature=temp)
        structured_llm = llm.with_structured_output(HypothesisVoteItem)
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", (
                "You are a rigorous scientific verification engine evaluating a research hypothesis.\n"
                "Evaluate the user's hypothesis based on the provided facts extracted from their uploaded draft paper.\n"
                "Your verdict must be one of:\n"
                "- 'SUPPORT': The facts strongly support this hypothesis.\n"
                "- 'REFUTE': The facts contradict or refute this hypothesis.\n"
                "- 'INSUFFICIENT_EVIDENCE': There is not enough evidence in the draft to support or refute this.\n\n"
                "Response must be structured in the requested format."
            )),
            ("user", "Hypothesis: {hypothesis}\n\nExtracted Facts:\n{context}")
        ])
        
        chain = prompt | structured_llm
        vote_item = await chain.ainvoke({
            "hypothesis": hypothesis,
            "context": citations_context
        })
        
        if not isinstance(vote_item, HypothesisVoteItem):
            raise TypeError(f"Expected HypothesisVoteItem, got {type(vote_item)}")
            
        detailed_votes.append(vote_item)
        v = vote_item.vote.upper().strip()
        if v in votes_map:
            votes_map[v] += 1
        else:
            votes_map["INSUFFICIENT_EVIDENCE"] += 1
            
    # 다수결 판정
    verdict = max(votes_map, key=lambda k: votes_map[k])
    consensus_ratio = votes_map[verdict] / 3.0
    
    return {
        "verdict": verdict,
        "consensus_ratio": consensus_ratio,
        "votes": votes_map,
        "detailed_votes": [v.model_dump() for v in detailed_votes]
    }

# 가설 검증 테스트
test_facts = [
    "We ran MicroAttention on a GPU server, reducing inference latency from 15ms to 10.5ms.",
    "MicroAttention uses only depthwise separable convolutions to save floating-point operations.",
    "We only tested classification; object detection was deferred to future work."
]
test_hypo = "MicroAttention reduces computation time on computer vision classification tasks."

verification = await verify_hypothesis_consistency(test_hypo, test_facts)
print(f"🎯 최종 판정: {verification['verdict']} (의견 일치율: {verification['consensus_ratio'] * 100:.1f}%)")
print(f"📊 투표 현황: {verification['votes']}")

/Users/pileuszu/Repos/bist-mini-2/backend/venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=HypothesisVoteItem(vote='...g with the hypothesis.'), input_type=HypothesisVoteItem])
  return self.__pydantic_serializer__.to_python(
/Users/pileuszu/Repos/bist-mini-2/backend/venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=HypothesisVoteItem(vote='...uces computation time.'), input_type=HypothesisVoteItem])
  return self.__pydantic_serializer__.to_python(


🎯 최종 판정: SUPPORT (의견 일치율: 100.0%)
📊 투표 현황: {'SUPPORT': 3, 'REFUTE': 0, 'INSUFFICIENT_EVIDENCE': 0}


/Users/pileuszu/Repos/bist-mini-2/backend/venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=HypothesisVoteItem(vote='...ng time in processing.'), input_type=HypothesisVoteItem])
  return self.__pydantic_serializer__.to_python(


## 4단계. 턴제 모의 디펜스 아레나 및 실시간 채점

디펜스 아레나는 질문과 답변을 주고받는 3턴짜리 턴제(Turn-based) 면접 세션입니다.
- **Turn 1 (디펜스 개시)**: 피어 리뷰 결과 중 가장 점수가 낮았던 에이전트의 피드백을 기반으로 공격 질문을 생성합니다.
- **Turns 2-3 (답변 제출 및 채점)**: 사용자의 해명을 4대 루브릭(논리적 엄밀성, 증거 제시, 비평 수용성, 전문성)에 근거해 실시간 채점(`ScoreDTO`)하고, 다음 압박 질문을 던집니다.
- **마지막 턴 종료 후**: 최종 심사 등급(Major Revision, Reject, Accept 등)을 포함한 최종 스코어카드 및 리포트를 도출합니다.

In [4]:
class ScoreDTO(BaseModel):
    score: int = Field(description="Defense response score out of 100")
    feedback: str = Field(description="Constructive critique on the strengths and gaps in their defense")

async def grade_user_response(question: str, user_answer: str) -> ScoreDTO:
    """사용자의 디펜스 답변을 실시간 채점합니다."""
    if not openai_key:
        return ScoreDTO(score=85, feedback="Good response but could use more statistical support.")
        
    llm = ChatOpenAI(model="gpt-4o-mini", openai_api_key=openai_key, temperature=0.2)
    structured_llm = llm.with_structured_output(ScoreDTO)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are an expert academic reviewer on an oral defense panel grading the author's response to your question.\n"
            "Analyze the question and the author's response. Grade the response strictly out of 100 based on the following rubric:\n"
            "1. Scientific Soundness & Logic (40%): Does the author address the core weakness logically without deflecting?\n"
            "2. Empirical Support & Evidence (30%): Does the author cite specific findings, experiments, or equations from their paper or known SOTA methods?\n"
            "3. Receptiveness to Criticism (20%): Does the author constructively acknowledge true limitations and outline realistic future mitigations?\n"
            "4. Professionalism (10%): Is the tone academic, respectful, and objective?\n\n"
            "Be highly critical. Do not give scores above 90 unless the response is flawless and mathematically or empirically backed.\n"
            "Provide a sharp, specific, and actionable critique detailing what was strong and what is still lacking in English."
        )),
        ("user", "Committee's Question: {question}\nAuthor's Defense Response: {answer}")
    ])
    
    chain = prompt | structured_llm
    grading = await chain.ainvoke({
        "question": question,
        "answer": user_answer
    })
    
    if not isinstance(grading, ScoreDTO):
        raise TypeError(f"Expected ScoreDTO, got {type(grading)}")
        
    return grading

# 디펜스 1턴 질문 채점 실습
q = "Why did you only compare MicroAttention with MobileNet baseline, and omit recent SOTA networks?"
ans = "We omitted recent SOTA networks because we had compute constraints. However, we plan to run comprehensive evaluations on ImageNet SOTA models in our future work as described in the draft."

score_result = await grade_user_response(q, ans)
print(f"🛡️ 디펜스 답변 점수: {score_result.score}/100")
print(f"💬 위원회 평가 피드백:\n{score_result.feedback}")

🛡️ 디펜스 답변 점수: 65/100
💬 위원회 평가 피드백:
The author acknowledges the limitation of not comparing MicroAttention with recent SOTA networks due to compute constraints, which is a reasonable explanation. However, the response lacks depth in addressing the core weakness of the study. Specifically, the author does not provide a logical justification for why the comparison with only MobileNet is sufficient or how it relates to the broader context of performance evaluation in the field. Furthermore, the mention of future plans to evaluate SOTA models is vague and does not offer any empirical support or evidence from the current work to substantiate the claims made. The author should have cited specific findings or methodologies that could have been compared with SOTA models, enhancing the credibility of their argument. Additionally, while the tone is respectful, it could benefit from a more academic and objective framing, particularly in acknowledging the implications of the omission on the study's

/Users/pileuszu/Repos/bist-mini-2/backend/venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=ScoreDTO(score=65, feedba... clearer path forward."), input_type=ScoreDTO])
  return self.__pydantic_serializer__.to_python(


## 5단계. 30분 미활동 세션 자동 소거 (Shredding & Wipe Out)

기밀 논문의 무단 잔존을 막기 위해 30분간 활동이 없는 세션의 PDF 파일 및 DB vector 테이블을 완전 소거하는 백엔드 Wipe Out의 개념을 학습합니다.

In [5]:
import shutil

def simulate_shredding(session_id: str):
    """활동 시간이 만료된 세션 리소스를 디스크 및 DB에서 연쇄 영구 완전 소거(Wipe Out)합니다."""
    # 1. 파일이 적재된 격리 격실 폴더 정의
    session_dir = f"./uploads/{session_id}"
    
    print(f"🔄 [Wipe Out] 세션 {session_id} 자동 소거를 가동합니다.")
    
    # 2. 물리 파일 제거
    try:
        shutil.rmtree(session_dir, ignore_errors=True)
        print(f"   - 물리 격실 디렉토리 완전 파쇄 완료: {session_dir}")
    except Exception as e:
        print(f"   - 파일 제거 실패: {e}")
        
    # 3. 데이터베이스 연쇄(Cascade) 셧다운 로직 설명
    print("""   - DB 쿼리 실행:
     DELETE FROM defense_arena_session WHERE session_id = :session_id;
     (ON DELETE CASCADE 규격에 의해 defense_arena_chunk 및 defense_history 레코드 자동 영구 소멸 완료)
    """)
    print("✅ 세션 리소스 영구 완전 소거 완료!")

simulate_shredding("session-uuid-1234")

🔄 [Wipe Out] 세션 session-uuid-1234 자동 소거를 가동합니다.
   - 물리 격실 디렉토리 완전 파쇄 완료: ./uploads/session-uuid-1234
   - DB 쿼리 실행:
     DELETE FROM defense_arena_session WHERE session_id = :session_id;
     (ON DELETE CASCADE 규격에 의해 defense_arena_chunk 및 defense_history 레코드 자동 영구 소멸 완료)
    
✅ 세션 리소스 영구 완전 소거 완료!
